# Problem-1. Parallel k-Means Clustering using Python (30 points)
Dataset: MNIST digits (subset of approximately 10,000 images), or sklearn.datasets.load_digits.
Use k = 10.

Tasks
Based on HW-1, Parallelize k-Means using Python threading. Divide the dataset among threads. Each thread processes a subset of data points during the E-step and computes local cluster sums/counts for the M-step. (10 points)
Parallelize k-Means using Python multiprocessing. Divide the dataset among processes. Each process computes assignments and local cluster sums/counts, which are combined to update the cluster centers. (10 points)

Test the sequential, threading, and multiprocessing versions using different numbers of workers, such as 1, 2, 4, and 8, and answer the following questions. (10 points)

In [6]:
import subprocess

subprocess.run(['python', 'hw2_p1.py'])

--- Benchmark Results ---
Sequential Execution Time: 0.2239 seconds

Threading Versions:
  Workers: 1 | Time: 0.2214 seconds
  Workers: 2 | Time: 0.3965 seconds
  Workers: 4 | Time: 0.4283 seconds
  Workers: 8 | Time: 0.5559 seconds
Workers: 2 | Results match 1-worker baseline? True
Workers: 4 | Results match 1-worker baseline? True
Workers: 8 | Results match 1-worker baseline? True

Multiprocessing Versions:
  Workers: 1 | Time: 0.3081 seconds
  Workers: 2 | Time: 0.2563 seconds
  Workers: 4 | Time: 0.1069 seconds
  Workers: 8 | Time: 0.1384 seconds
Workers: 2 | Results match 1-worker baseline? True
Workers: 4 | Results match 1-worker baseline? True
Workers: 8 | Results match 1-worker baseline? True


CompletedProcess(args=['python', 'hw2_p1.py'], returncode=0)

(see hw2_p1.py)

(a) Does increasing the number of Python threads make the program run faster? <br>
ANSWER: Increasing the threads causes the program runtime to increase instead of decrease. This is likely because creating multiple threads results in each thread executing concurrently, not being able to achieve true parallel execution.

<br>

(b) Does increasing the number of Python processes make the program run faster?<br>
ANSWER: Increasing the number of processes causes the program runtime to decrease as more workers are added. This is because processes were able to run in parallel as opposed to threads that have to run concurrently.

<br>

(c) Explain your results based on Python's Global Interpreter Lock (GIL).<br>
ANSWER: Python's GIL is a mutex which means that in the multithreading results, threads were forced to run sequentially due to the GIL only allowing one thread to run on the CPU at a time. In the multiprocessing case, Python processes are a workaround to the GIL because there is a unique GIL for each Python process, as there is a corresponding dedicated Python interpreter for each thread.

<br>

(d) Does changing the number of workers affect the final clustering result?<br>
ANSWER: Given the tests done, changing the number of workers does not affect the final clustering result in any case for multithreading or multiprocessing.

# Problem-2. Parallel Nonlinear Regression using Python Multiprocessing (40 points)
Use the provided NonlinearRegression.py and 20K_Datapoints.csv in HW1.

The original program uses full-batch gradient descent. At each epoch, divide the training data among processes. Each process calculates gradients for its local data. The partial gradients are then combined before updating the model parameters.

Implement the following versions:

Run the provided sequential version and record its entire runtime. (4 points)

Implement a multiprocessing version using shared memory and multiprocessing.Lock. Use the Lock when processes combine their local gradients into shared gradients. (9 points)

Implement the training using multiprocessing.Pool. Each worker calculates local gradients and returns them to the main process. (9 points)

Implement the training using multiprocessing.Queue. Each process sends its local gradients to the main process through a Queue. (7 points)

Implement the training using multiprocessing.Pipe. Each process sends its local gradients to the main process through a Pipe. (7 points)

Compare the entire runtime of the sequential, Lock, Pool, Queue, and Pipe implementations. Answer the following questions. (4 points)

Hints
<br>
Distribute the data among processes at the beginning. During every epoch, each process calculates its local gradients. The gradients are combined, the model is updated, and the updated model is used in the next epoch.

This is synchronized parallel training. We do not consider mini-batch training in this assignment.

Please use NumPy for numerical computation.

In [1]:
import time
import subprocess

begin = time.time()
subprocess.run(['python', 'NonlinearRegression_Sequential.py'])
print("Time taken for sequential version to finish: ", time.time() - begin)

Figure(640x480)
Final Model Parameters: a=9.502181674800925, b=11.861262607997416, c=10.756680205787243
Figure(640x480)
Time taken for sequential version to finish:  17.937299966812134


In [2]:
subprocess.run(['python', 'hw2_p2.py', 'shared_memory'])
subprocess.run(['python', 'hw2_p2.py', 'pool'])
subprocess.run(['python', 'hw2_p2.py', 'queue'])
subprocess.run(['python', 'hw2_p2.py', 'pipe'])

Running method: SHARED_MEMORY with 4 processes...
[Shared Memory] Runtime: 0.0706 seconds
Final Model Parameters: a=10.36911457555485, b=5.989114913608169, c=2.804321248565662
Figure(640x480)
Running method: POOL with 4 processes...
[Pool] Runtime: 0.1652 seconds
Final Model Parameters: a=10.36911457555485, b=5.989114913608169, c=2.8043212485656612
Figure(640x480)
Running method: QUEUE with 4 processes...
[Queue] Runtime: 0.1674 seconds
Final Model Parameters: a=10.36911457555485, b=5.989114913608169, c=2.804321248565662
Figure(640x480)
Running method: PIPE with 4 processes...
[Pipe] Runtime: 0.0947 seconds
Final Model Parameters: a=10.36911457555485, b=5.989114913608169, c=2.8043212485656612
Figure(640x480)


CompletedProcess(args=['python', 'hw2_p2.py', 'pipe'], returncode=0)

\*\*please see hw2_p2.py for the implementations

(a) Why is synchronization needed when multiple processes update shared gradients?<br>
ANSWER: Synchronization prevents race conditions when multiple processes are accessing the same shared gradients. Processes would likely overwrite each other if there was not synchronization.

<br>

(b) Why can frequent use of a Lock reduce performance?<br>
ANSWER: Frequent use of a lock would create a bottleneck in performance. Unnecessary time would be taken in order to access the lock as opposed to running in parallel.

<br>

(c) Compare the performance of Pool, Queue, Pipe, and shared memory.<br>
ANSWER: The results of each of the methods do not differ. Using 4 processes across the methods, the runtime performance differs slightly but relatively are the same. Shared memory performs the fastest and pooling performs the slowest. 

<br>

(d) Do the parallel implementations produce approximately the same model parameters and loss as the sequential implementation?<br>
ANSWER: The parallel implementations produce the same exact model parameters matching the parallel implementations.


# Problem-3. Parallel Nonlinear Regression using OpenMP (40 points)
Use the same nonlinear regression problem and dataset from Problem-2.

Implement a shared-memory parallel version using one  (not both) of the following:

C/C++ with OpenMP, or
Python with PyOMP / OMP4Py.
Tasks
Parallelize the gradient computation in each training epoch. Different threads should process different training samples. (20 points)
Conceptually:

for each epoch:
    initialize gradients

    parallel for each training sample:
        calculate prediction
        calculate error
        calculate gradient contribution

    combine gradients
    update model parameters
For C/C++, you may use OpenMP constructs such as:

#pragma omp parallel for reduction(...)
Use appropriate shared, private, and reduction/local partial variables.
Test your program using 1, 2, 4, and 8 threads. Report runtime and speedup. (10 points)
<br>

In [4]:
subprocess.run(['python', 'hw2_p3.py', 'pool'])

Threads   Runtime (s)    Speedup     Final Parameters (a, b, c)
-----------------------------------------------------------------
1         0.4365         1.00        a=2.4319, b=1.3199, c=2.7595
2         0.1912         2.28        a=2.4319, b=1.3199, c=2.7595
4         0.1978         2.21        a=2.4319, b=1.3199, c=2.7595
8         0.1915         2.28        a=2.4319, b=1.3199, c=2.7595


CompletedProcess(args=['python', 'hw2_p3.py', 'pool'], returncode=0)

Compare the OpenMP implementation with the sequential and Python multiprocessing implementations from Problem-2. 
Answer the following questions. (10 points)
<br>
<br>

\*\*please see hw2_p3.py for the implementations


(a) Which variables should be shared and which should be private?<br>
ANSWER: Individual predictions and local gradients should be private and model parameters and training set arrays should be shared among threads.

<br>

(b) Why can gradient contributions from different training samples be calculated in parallel?<br>
ANSWER: Each training sample's gradient contribution depends only on the current model parameters and its own data points, meaning its computation is completely independent of other samples within the same epoch.

<br>

(c) How are gradients from different threads combined?<br>
ANSWER: Gradients from different threads are combined using an OpenMP reduction operation, which safely aggregates and sums the thread-local gradient totals into global gradient accumulators.

<br>

(d) Does increasing the number of threads always make the program faster?<br>
ANSWER: No, increasing the number of threads does not always improve performance as shown in the results. The change in performance plateaus due to thread management.

<br>

(e) Compare the performance of OpenMP and Python multiprocessing.<br>
ANSWER: OpenMP's runtime performance is slightly slower than python multiprocessing. The fastest the OpenMP implementation can achieve is 0.1912 seconds wheareas all multiprocessing methods were faster than that. 